# Sparse Retrieval: from BM25 to SPLADE — Practical Lab

Companion to the lecture **Sparse Information Retrieval v2**.

| Block | Topic | Time |
|---|---|---|
| 0 | Setup and toy corpus | 3′ |
| 1 | BM25 and the vocabulary mismatch problem | 10–12′ |
| 2 | How SPLADE works | 12–15′ |
| 3 | The same example with SPLADE | 12–15′ |
| 4 | Evaluation on standard datasets | 12–15′ |

Legend: 🔮 **predict** before running · ✂️ optional · ✏️ **try at home**

The thread of the lab: **BM25 and SPLADE produce the same kind of object**, a sparse vector over a vocabulary.
BM25 fills it with a fixed formula and only the words in the text; SPLADE **learns** the weights and can activate words that are **not** in the text.

---
## Block 0 · Setup

In [ ]:
!pip -q install "sentence-transformers>=5.4.1" nltk datasets

In [ ]:
import re, math
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch

pd.set_option("display.precision", 2)
pd.set_option("display.max_columns", 40)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

### The toy corpus
Six short documents and one query. Only **D0** really answers the query.

In [ ]:
DOCS = [
    "Repairing a punctured bicycle wheel: remove the inner tube, patch the hole and inflate it again.",  # D0 relevant
    "The car tire looked flat, so we stopped at a gas station to check it.",                            # D1 decoy
    "Fix the bug in the login page before the release.",                                                # D2
    "A recipe for flat bread with olive oil and rosemary.",                                             # D3
    "The weather in Rome will be sunny all week.",                                                      # D4
    "Tire pressure should be checked once a month on every car.",                                       # D5
]
DOC_IDS = [f"D{i}" for i in range(len(DOCS))]
QUERY = "How do I fix a flat tire on my bike?"

for i, d in zip(DOC_IDS, DOCS):
    print(f"{i}: {d}")
print(f"\nQUERY: {QUERY}")

---
## Block 1 · BM25 and the vocabulary mismatch problem

### 1.1 Tokenization
BM25 works on **words**: lowercase, split, drop stopwords (very common words that carry little meaning).

In [ ]:
STOPWORDS = {"a", "an", "the", "and", "or", "of", "to", "in", "on", "at", "for", "with", "so", "we", "it",
             "is", "be", "do", "i", "my", "how", "before", "should", "every", "once", "will", "all", "again"}

def tokenize(text):
    return [w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOPWORDS]

doc_tokens = [tokenize(d) for d in DOCS]
query_tokens = tokenize(QUERY)

for i, toks in zip(DOC_IDS, doc_tokens):
    print(f"{i}: {toks}")
print(f"\nQUERY: {query_tokens}")

### 1.2 The BM25 representation
For each document and each term $t$, BM25 computes a weight:

$$w(t, d) = \text{IDF}(t)\cdot\frac{tf(t,d)\,(k_1+1)}{tf(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}
\qquad \text{IDF}(t)=\ln\left(1+\frac{N-n_t+0.5}{n_t+0.5}\right)$$

- $tf$: how many times $t$ appears in $d$ · $n_t$: how many documents contain $t$ · $N$: number of documents
- $|d|$: document length · avgdl: average length · $k_1$, $b$: parameters (typically 1.2 and 0.75)

In [ ]:
class BM25:
    """A minimal BM25, written to be read: every weight comes from the formula above."""

    def __init__(self, tokenized_docs, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.docs = tokenized_docs
        self.N = len(tokenized_docs)
        self.avgdl = np.mean([len(d) for d in tokenized_docs])
        self.vocab = sorted({t for d in tokenized_docs for t in d})
        df = Counter(t for d in tokenized_docs for t in set(d))            # n_t: documents containing t
        self.idf = {t: math.log(1 + (self.N - df[t] + 0.5) / (df[t] + 0.5)) for t in self.vocab}

    def term_weight(self, t, doc):
        tf = doc.count(t)
        if tf == 0 or t not in self.idf:
            return 0.0
        norm = self.k1 * (1 - self.b + self.b * len(doc) / self.avgdl)     # length normalization
        return self.idf[t] * tf * (self.k1 + 1) / (tf + norm)              # saturated TF x IDF

    def matrix(self):
        """Documents x vocabulary table of BM25 weights."""
        return pd.DataFrame([[self.term_weight(t, d) for t in self.vocab] for d in self.docs],
                            columns=self.vocab)

    def score(self, query_tokens, doc):
        return sum(self.term_weight(t, doc) for t in query_tokens)

bm25 = BM25(doc_tokens)
W = bm25.matrix()
W.index = DOC_IDS
print(f"{W.shape[0]} documents x {W.shape[1]} vocabulary terms · "
      f"{(W > 0).values.mean():.0%} of the cells are non-zero")
W.style.background_gradient(cmap="Greens", axis=None).format(lambda v: f"{v:.2f}" if v else "")

Each **row is a sparse vector over the vocabulary**: a document only activates the words it contains.
Keep this picture in mind: SPLADE will produce exactly this kind of vector.

### 1.3 IDF: rare words weigh more

In [ ]:
idf = pd.Series(bm25.idf).sort_values()
print("Most common terms (low IDF):\n", idf.head(4).round(2).to_string())
print("\nRarest terms (high IDF):\n", idf.tail(4).round(2).to_string())

### 1.4 TF saturation and length normalization
🔮 **Predict:** if a word appears 20 times instead of once, is the document 20 times more relevant?

In [ ]:
tfs = np.arange(0, 21)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))

# TF saturation: same document length, growing term frequency
for k1 in [0.5, 1.2, 3.0]:
    ax1.plot(tfs, [tf * (k1 + 1) / (tf + k1) for tf in tfs], label=f"k1 = {k1}")
ax1.plot(tfs, tfs / 2, "--", c="gray", label="raw tf (scaled)")
ax1.set_xlabel("term frequency in the document"); ax1.set_ylabel("TF component"); ax1.set_ylim(0, 4.5)
ax1.set_title("TF saturation: repeating a word helps less and less"); ax1.legend()

# Length normalization: same tf = 1, growing document length
ratios = np.linspace(0.25, 4, 50)                                           # |d| / avgdl
for b in [0.0, 0.75, 1.0]:
    k1 = 1.2
    ax2.plot(ratios, [(k1 + 1) / (1 + k1 * (1 - b + b * r)) for r in ratios], label=f"b = {b}")
ax2.set_xlabel("document length / average length"); ax2.set_ylabel("TF component (tf = 1)")
ax2.set_title("Length normalization: one match in a long document counts less"); ax2.legend()
plt.tight_layout(); plt.show()

### 1.5 The inverted index
The same table, read **by columns**: for every term, the documents that contain it.
At query time we only touch the lists of the query terms, not the whole collection: this is why BM25 is fast.

In [ ]:
inverted_index = {t: [(DOC_IDS[i], round(w, 2)) for i, w in enumerate(W[t]) if w > 0] for t in W.columns}

for t in ["tire", "flat", "fix", "car", "wheel"]:
    print(f"{t:>6} -> {inverted_index.get(t, [])}")

### 1.6 Scoring the query
The query is also a vector over the vocabulary, but **binary**: a term is either in the query or not.
Score = sum of the document's weights for the query terms.

🔮 **Predict:** which document wins? What is the score of D0, the only relevant one?

In [ ]:
def bm25_ranking(query_tokens):
    rows = []
    for i, doc in zip(DOC_IDS, bm25.docs):
        contributions = {t: round(bm25.term_weight(t, doc), 2) for t in query_tokens if bm25.term_weight(t, doc) > 0}
        rows.append({"doc": i, "BM25 score": bm25.score(query_tokens, doc), "matched terms": contributions,
                     "text": DOCS[DOC_IDS.index(i)][:60]})
    return pd.DataFrame(rows).sort_values("BM25 score", ascending=False).reset_index(drop=True)

print("query terms:", query_tokens)
bm25_ranking(query_tokens)

### 1.7 The vocabulary mismatch problem
D0 is the answer, but it scores **0**: it says *repairing / punctured / bicycle / wheel*, the query says *fix / flat / bike / tire*.
Look at the representation: the query's columns and D0's columns **do not overlap at all**.

In [ ]:
cols = query_tokens + [t for t in doc_tokens[0] if t not in query_tokens]
view = W.reindex(columns=cols, fill_value=0).loc[["D0"]]
view.loc["QUERY"] = [1 if t in query_tokens else 0 for t in cols]
view.style.background_gradient(cmap="Greens", axis=None).format(lambda v: f"{v:.2f}" if v else "")

🔮 **Does stemming help?** Reduce words to a common root (*repairing → repair*).

In [ ]:
from nltk.stem import PorterStemmer
stem = PorterStemmer().stem

q_stems = {stem(t) for t in query_tokens}
d0_stems = {stem(t) for t in doc_tokens[0]}
print("query stems:", sorted(q_stems))
print("D0 stems   :", sorted(d0_stems))
print("overlap    :", q_stems & d0_stems or "none")

Stemming fixes *tire / tires* or *repair / repairing*, but **not synonyms**: *fix* ≠ *repair*, *bike* ≠ *bicycle*.

**The question for the next block:** what if the vector could also activate words that are **not in the text**?

✏️ **Try at home:** change `k1` and `b` in `BM25(...)` and see how the ranking changes. Add a document with *tire* repeated 10 times.

---
## Block 2 · How SPLADE works

**SPLADE** (SParse Lexical AnD Expansion) produces the same object as BM25, a sparse vector over a vocabulary
(BERT's 30,522 WordPiece tokens), with two differences:
- the weights are **learned**, not computed with a formula
- the vector can activate **expansion terms**: words that are not in the text but are related to it

The trick: reuse the **masked language modeling (MLM) head** of BERT.

### 2.1 The MLM head already knows related words
🔮 **Predict:** what will BERT put in the blank?

In [ ]:
from transformers import pipeline

fill = pipeline("fill-mask", model="bert-base-uncased", device=device)
for p in fill("I need to fix the flat [MASK] on my bike.", top_k=8):
    print(f"{p['token_str']:>10}  {p['score']:.3f}")

The MLM head gives a score to **every word in the vocabulary**, at **every position**, not only at `[MASK]`.
Let's look at what it predicts for each token of the query.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_mlm = AutoModelForMaskedLM.from_pretrained("bert-base-uncased").to(device).eval()

inputs = bert_tok(QUERY, return_tensors="pt").to(device)
with torch.no_grad():
    logits = bert_mlm(**inputs).logits                    # (batch, tokens, vocabulary)
print("MLM logits shape:", tuple(logits.shape), "-> one score per vocabulary word, for each token\n")

for pos, tok_id in enumerate(inputs.input_ids[0]):
    top = logits[0, pos].topk(6).indices
    print(f"{bert_tok.decode([tok_id]):>8} -> {bert_tok.convert_ids_to_tokens(top)}")

For the token *bike* the head also scores *bicycle*, *car*…: the **expansion** comes for free.
SPLADE turns these (tokens × vocabulary) scores into **one** vector per text.

### 2.2 Sparse tensors
A vocabulary-sized vector is huge (30k–250k dimensions) but almost all zeros, so we store only the non-zero entries.

In [ ]:
dense = torch.tensor([
    [0.0, 0.0, 1.7, 0.0, 0.0, 2.3],
    [0.0, 5.1, 0.0, 0.0, 0.0, 0.0],
])
sparse = dense.to_sparse()
print(sparse)
print(f"\nstored values: {sparse._nnz()} instead of {dense.numel()}")

### 2.3 SPLADE pooling: from (tokens × vocabulary) to one vector
`SpladePooling` from sentence-transformers applies three operations to the MLM logits:
1. **ReLU**: keep only positive evidence
2. **log1p**: damp large values (the same idea as BM25's TF saturation)
3. **max over tokens**: one score per vocabulary word, the strongest across the text

A toy example: 1 text, 3 token positions, a vocabulary of 4 words.

🔮 **Predict:** token 3 has logits of 10 everywhere, but it is **padding** (`attention_mask = 0`). Does it affect the result?

In [ ]:
from sentence_transformers.sparse_encoder.modules import SpladePooling

pool = SpladePooling(pooling_strategy="max", activation_function="relu")
pool.eval()

features = {
    "token_embeddings": torch.tensor([
        [  # batch item 0, shape (3 tokens, 4 vocabulary words)
            [ 1.0, -1.0,  0.0, 2.0],   # token 1 logits
            [ 0.5,  3.0, -2.0, 1.0],   # token 2 logits
            [10.0, 10.0, 10.0, 10.0],  # token 3 logits, but masked out below
        ]
    ]),
    "attention_mask": torch.tensor([[1, 1, 0]]),
}
print("max pooling:", pool(features)["sentence_embedding"])

In [ ]:
# Same logits, sum instead of max (SPLADE v1 used sum, SPLADE v2 switched to max)
pool_sum = SpladePooling(pooling_strategy="sum", activation_function="relu").eval()
print("sum pooling:", pool_sum(features)["sentence_embedding"])

### The formula behind the code
With MLM logits $L \in \mathbb{R}^{B\times T\times V}$ and attention mask $M \in \{0,1\}^{B\times T}$:

$$S_{b,v} = \max_{t} \; \log\big(1 + \text{ReLU}(L_{b,t,v} \cdot M_{b,t})\big)$$

(sum pooling replaces $\max_t$ with $\sum_t$). The result $S \in \mathbb{R}^{B\times V}$ has one weight per vocabulary word,
and every word with no positive evidence stays exactly **0**.

### 2.4 The real pipeline
A `SparseEncoder` = **MLM transformer** + **SpladePooling**. Here built on top of `mmBERT-base`, a multilingual encoder
that has **not** been trained for retrieval: we use it to look at the shapes.

In [ ]:
from sentence_transformers import SparseEncoder

untrained = SparseEncoder("jhu-clsp/mmBERT-base", device=device)
print(untrained)

features = untrained.preprocess(["hello words", "hello worlds"])
features = {k: v.to(device) if torch.is_tensor(v) else v for k, v in features.items()}
with torch.no_grad():
    out = untrained(features)

print("\ninput_ids          :", tuple(out["input_ids"].shape), " (texts, tokens)")
print("token_embeddings   :", tuple(out["token_embeddings"].shape), " (texts, tokens, vocabulary) = MLM logits")
print("sentence_embedding :", tuple(out["sentence_embedding"].shape), " (texts, vocabulary) = after SpladePooling")

The architecture is ready, but the weights are not: this model was trained to fill blanks, not to retrieve.
In Block 3 we use the **same architecture** after SPLADE training, and see the difference.

---
## Block 3 · The same example with SPLADE

Same corpus, same query, same steps as Block 1. Model: [`naver/splade-cocondenser-ensembledistil`](https://huggingface.co/naver/splade-cocondenser-ensembledistil), a BERT-base SPLADE model trained on MS MARCO.

### 3.1 Encode queries and documents
Queries and documents have separate methods, because some sparse models encode them differently.

In [ ]:
splade = SparseEncoder("naver/splade-cocondenser-ensembledistil", device=device)

doc_emb = splade.encode_document(DOCS)            # sparse tensor (documents, 30522)
q_emb = splade.encode_query([QUERY])              # sparse tensor (1, 30522)
print("documents:", tuple(doc_emb.shape), "· query:", tuple(q_emb.shape), "·", doc_emb.layout)

### 3.2 The SPLADE representation
The same documents × vocabulary table as Block 1, now with SPLADE weights (top 12 terms per document).

In [ ]:
decoded_docs = splade.decode(doc_emb, top_k=12)   # list of [(token, weight), ...] per document
S = pd.DataFrame([dict(terms) for terms in decoded_docs], index=DOC_IDS).fillna(0)
S = S[S.max().sort_values(ascending=False).index]  # strongest terms first
print(f"{S.shape[1]} vocabulary terms in this view (top 12 per document)")
S.style.background_gradient(cmap="Purples", axis=None).format(lambda v: f"{v:.2f}" if v else "")

### 3.3 Inside one document: words in the text vs expansion terms
🔮 **Predict:** which words that are **not** in D0 will SPLADE activate?

In [ ]:
def explain(text, is_query=False, top_k=20):
    """Top SPLADE terms of one text, marking which ones appear in it."""
    emb = splade.encode_query([text]) if is_query else splade.encode_document([text])
    in_text = set(splade.tokenizer.tokenize(text))
    rows = [{"term": t, "weight": w, "in text?": "yes" if t in in_text else "EXPANSION"}
            for t, w in splade.decode(emb, top_k=top_k)[0]]
    return pd.DataFrame(rows)

print("D0:", DOCS[0])
explain(DOCS[0])

### 3.4 Inside the query
In BM25 the query vector was **binary**. In SPLADE it has **learned weights**, expansion terms,
and stopwords like *how* and *do* get low or zero weight.

In [ ]:
print("QUERY:", QUERY)
explain(QUERY, is_query=True)

### 3.5 Score and rank
Score = dot product between the query vector and each document vector (exactly like BM25: a sum over shared terms).

🔮 **Predict:** does D0 still score 0?

In [ ]:
splade_scores = splade.similarity(q_emb, doc_emb)[0].cpu()
bm25_scores = {i: bm25.score(query_tokens, d) for i, d in zip(DOC_IDS, bm25.docs)}

ranking = pd.DataFrame({"doc": DOC_IDS,
                        "BM25 score": [bm25_scores[i] for i in DOC_IDS],
                        "SPLADE score": splade_scores.tolist(),
                        "text": [d[:60] for d in DOCS]})
ranking["BM25 rank"] = ranking["BM25 score"].rank(ascending=False, method="min").astype(int)
ranking["SPLADE rank"] = ranking["SPLADE score"].rank(ascending=False, method="min").astype(int)
ranking.sort_values("SPLADE score", ascending=False).reset_index(drop=True)

### 3.6 Why does D0 match now?
The score is a sum over the terms that query and document share: query weight × document weight.

In [ ]:
def match_breakdown(query, doc, top_k=15):
    """Shared terms between a query and a document, and how much each adds to the score."""
    q_terms = dict(splade.decode(splade.encode_query([query]))[0])
    d_terms = dict(splade.decode(splade.encode_document([doc]))[0])
    shared = [(t, q_terms[t], d_terms[t], q_terms[t] * d_terms[t]) for t in q_terms.keys() & d_terms.keys()]
    df = pd.DataFrame(shared, columns=["term", "query weight", "doc weight", "contribution"])
    return df.sort_values("contribution", ascending=False).head(top_k).reset_index(drop=True)

bd = match_breakdown(QUERY, DOCS[0])
print(f"shared terms contribute {bd.contribution.sum():.2f} (top {len(bd)} shown)")
bd

In Block 1 the query and D0 had **no** column in common. Now they meet on expansion terms:
the vocabulary mismatch is solved **inside the representation**, with the same scoring and the same inverted index as BM25.

### 3.7 Sparsity: how many terms are active?
🔮 **Predict:** how many of the 30,522 dimensions does SPLADE activate for a document? And the untrained mmBERT?

In [ ]:
def active_dims(emb):
    return (emb.to_dense() != 0).sum(dim=1).float()

bm25_active = (W > 0).sum(axis=1).astype(float)
untrained_doc = untrained.encode_document(DOCS)
untrained_q = untrained.encode_query([QUERY])

sparsity = pd.DataFrame({
    "vocabulary size": [len(bm25.vocab), splade.get_sentence_embedding_dimension(), untrained.get_sentence_embedding_dimension()],
    "active dims / document (mean)": [bm25_active.mean(), active_dims(doc_emb).mean().item(), active_dims(untrained_doc).mean().item()],
    "active dims / query": [len(set(query_tokens)), active_dims(q_emb).item(), active_dims(untrained_q).item()],
}, index=["BM25", "SPLADE (trained)", "mmBERT (untrained)"])
sparsity["% of vocabulary (document)"] = 100 * sparsity["active dims / document (mean)"] / sparsity["vocabulary size"]
sparsity.round(1)

- **BM25**: only the words in the text.
- **Trained SPLADE**: the words in the text **plus** a controlled number of expansions.
- **Untrained mmBERT**: the same architecture activates a large part of the vocabulary: not sparse, and not meaningful.

What makes the difference is **training**: a ranking loss (positive vs negative documents) plus the **FLOPS regularization**,
which penalizes active dimensions and pushes most weights to exactly zero.
Fewer active terms = shorter posting lists in the inverted index = cheaper search.

✏️ **Try at home**
- Write a query with **no** word in common with D0 (e.g. *"my bicycle has a hole in the tube"*, then *"patch inner tube"*) and compare the rankings.
- Try an **Italian** query with this English model: what happens to the tokens and the expansions?

---
## Block 4 · Evaluation on standard datasets

A toy example proves nothing. We now evaluate BM25 and SPLADE on standard retrieval benchmarks (from BEIR / MTEB)
with **nDCG@10**: are the relevant documents at the top of the ranking?

### 4.1 Anatomy of a retrieval benchmark
Every retrieval dataset has three parts:
- **corpus**: document id → text
- **queries**: query id → text
- **qrels** (relevance judgments): query id → relevant document ids, with a relevance grade

Example: [SciFact](https://huggingface.co/datasets/mteb/scifact) (scientific claims → abstracts that support or refute them).

In [ ]:
from datasets import load_dataset

corpus = load_dataset("mteb/scifact", "corpus", split="corpus")
queries = load_dataset("mteb/scifact", "queries", split="queries")
qrels = load_dataset("mteb/scifact", "default", split="test")
print(f"corpus: {len(corpus):,} documents · queries: {len(queries):,} · test judgments: {len(qrels):,}")

# One query and its relevant document
q = qrels[0]
query_text = next(r["text"] for r in queries if r["_id"] == str(q["query-id"]))
doc = next(r for r in corpus if r["_id"] == str(q["corpus-id"]))
print(f"\nQUERY  : {query_text}")
print(f"RELEVANT (score {q['score']}): {doc['title']}\n  {doc['text'][:300]}...")

### 4.2 Evaluate with our library
<!-- TODO: replace with the library name, link and install command -->
The library adapts MTEB to sparse models like SPLADE.

In [ ]:
# TODO: install and import the library
# !pip -q install <library>
# import <library>

In [ ]:
# TODO: evaluate SPLADE on SciFact and NFCorpus
# MODELS = ["naver/splade-cocondenser-ensembledistil"]
# TASKS = ["SciFact", "NFCorpus"]
# results = ...

In [ ]:
# TODO: evaluate the BM25 baseline on the same tasks
# bm25_results = ...

### 4.3 Read the results
🔮 **Predict:** will SPLADE beat BM25 on **every** dataset?

In [ ]:
# TODO: collect nDCG@10 (and Recall@100) into one table: rows = datasets, columns = models
# table = pd.DataFrame(...)
# table

**Reading nDCG@10**
- 1.0 = every relevant document at the top, in the ideal order · 0 = none in the top 10.
- BEIR's message: learned models win **on average**, but BM25 stays strong on some domains,
  especially specialized vocabularies the model never saw in training.

### 4.4 Back to one query
Pick a query where SPLADE beats BM25 and use `match_breakdown` from Block 3 to see **which terms** made the difference.

In [ ]:
# TODO: pick a query id from the results where SPLADE > BM25
# q_text, d_text = ..., ...
# match_breakdown(q_text, d_text)

✏️ **Homework**
- **H1** Evaluate a third model (e.g. the untrained mmBERT, or an inference-free SPLADE) on one task. Explain its score using its sparsity.
- **H2** Find 3 queries where BM25 beats SPLADE and 3 where SPLADE wins. Explain each with the term breakdown.
- **H3** Evaluate on an Italian retrieval task: English SPLADE vs a multilingual sparse model vs BM25.